# How much GPU memory does an LLM really need?

A hands-on benchmark of GPU memory, speed and answer quality for a small open language model.

**What this notebook measures**
1. **Baseline:** the model in 16-bit precision
2. **Quantization:** the same model in 8-bit and 4-bit
3. **KV cache:** how memory grows as prompts get longer
4. **Batching:** how processing several prompts together changes throughput

**Before you start:** in Colab, go to **Runtime → Change runtime type → T4 GPU → Save**. Then run the cells from top to bottom with **Shift + Enter**.

## Step 0: Install libraries and check the GPU

`transformers` loads the model, `accelerate` places it on the GPU, and `bitsandbytes` handles 8-bit and 4-bit quantization.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes tabulate
!nvidia-smi

**Important:** when this install cell finishes, click **Runtime → Restart session**, then continue with the next cell. Colab needs the restart to use the newly installed libraries; without it, 8-bit and 4-bit loading can fail with a "requires bitsandbytes" error.

## Step 1: Settings and helper functions

We use **Qwen2.5 1.5B Instruct**, a small open model that downloads without a login and fits easily on a free T4 GPU (about 15 GB of memory).

* `load_model` loads the model in a chosen precision
* `run_prompts` generates answers and measures **peak GPU memory** and **tokens per second**
* `time_to_first_token` measures how long until the first word appears
* `free_memory` clears the GPU so each experiment starts clean

In [ ]:
import importlib.metadata as _md
try:
    print("bitsandbytes", _md.version("bitsandbytes"))
except _md.PackageNotFoundError:
    raise SystemExit("bitsandbytes is missing: run the install cell, then Runtime -> Restart session.")
import gc, time, torch
import pandas as pd
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_NEW_TOKENS = 64
GB = 1024 ** 3

assert torch.cuda.is_available(), "No GPU found. Set Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
print("Total GPU memory: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / GB))

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"   # needed for batched generation

# A small fixed test set, so we can compare answer quality between setups
TEST_PROMPTS = [
    "What is the capital of Germany? Answer in one word.",
    "What is 17 multiplied by 3? Answer with the number only.",
    "Explain in one sentence what a GPU is.",
    "Name three primary colors.",
    "Translate 'good morning' into German.",
]
EXPECTED = ["berlin", "51", None, None, "guten morgen"]   # None = judge by reading

def free_memory():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

def load_model(precision):
    free_memory()
    if precision == "16-bit":
        model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="cuda")
    elif precision == "8-bit":
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID, quantization_config=BitsAndBytesConfig(load_in_8bit=True), device_map="cuda")
    else:  # 4-bit
        model = AutoModelForCausalLM.from_pretrained(
            MODEL_ID,
            quantization_config=BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16),
            device_map="cuda")
    model.eval()
    return model

def build_inputs(prompts):
    texts = [tokenizer.apply_chat_template([{"role": "user", "content": p}],
                                           tokenize=False, add_generation_prompt=True) for p in prompts]
    return tokenizer(texts, return_tensors="pt", padding=True).to("cuda")

@torch.no_grad()
def run_prompts(model, prompts, max_new_tokens=MAX_NEW_TOKENS):
    """Generate answers for a batch of prompts and measure memory and speed."""
    torch.cuda.reset_peak_memory_stats()
    inputs = build_inputs(prompts)
    torch.cuda.synchronize(); start = time.perf_counter()
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize(); seconds = time.perf_counter() - start
    new_tokens = out[:, inputs["input_ids"].shape[1]:]
    n_tokens = int((new_tokens != tokenizer.eos_token_id).sum())
    return {
        "answers": tokenizer.batch_decode(new_tokens, skip_special_tokens=True),
        "peak_gb": torch.cuda.max_memory_allocated() / GB,
        "seconds": seconds,
        "tokens": n_tokens,
        "tokens_per_sec": n_tokens / seconds,
    }

@torch.no_grad()
def time_to_first_token(model, prompt):
    inputs = build_inputs([prompt])
    torch.cuda.synchronize(); start = time.perf_counter()
    model.generate(**inputs, max_new_tokens=1, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    torch.cuda.synchronize()
    return time.perf_counter() - start

def quality_score(answers):
    checked = [(a, e) for a, e in zip(answers, EXPECTED) if e is not None]
    return round(sum(e in a.lower() for a, e in checked) / len(checked), 2)

## Step 2: Predict before you measure

A model's weights need roughly **number of parameters × bytes per number**:

| Precision | Bytes per number |
|---|---|
| 16-bit | 2 |
| 8-bit | 1 |
| 4-bit | 0.5 |

Calculate the prediction first, then compare it with the measurement in the next phase.

In [ ]:
_m = load_model("16-bit")
n_params = sum(p.numel() for p in _m.parameters())
del _m
free_memory()
print(f"Parameters: {n_params/1e9:.2f} billion")
for name, b in [("16-bit", 2), ("8-bit", 1), ("4-bit", 0.5)]:
    print(f"Predicted weight memory in {name}: {n_params * b / GB:.2f} GB")

## Phase 1 and 2: Compare 16-bit, 8-bit and 4-bit

For each precision we record:
* **weights_gb:** memory used just by loading the model
* **peak_gb:** highest memory during generation (weights + KV cache + working space)
* **tokens_per_sec:** generation speed
* **first_token_s:** seconds until the first word appears
* **quality:** share of checkable test questions answered correctly

This takes a few minutes.

In [ ]:
results, all_answers = [], {}
model = None
for precision in ["16-bit", "8-bit", "4-bit"]:
    if model is not None:
        del model
    model = load_model(precision)
    weights_gb = torch.cuda.memory_allocated() / GB
    run_prompts(model, TEST_PROMPTS[:1], max_new_tokens=8)   # warm-up, not measured
    ttft = time_to_first_token(model, TEST_PROMPTS[0])
    answers, peaks, speeds = [], [], []
    for p in TEST_PROMPTS:                                    # one prompt at a time
        r = run_prompts(model, [p])
        answers.append(r["answers"][0]); peaks.append(r["peak_gb"]); speeds.append(r["tokens_per_sec"])
    all_answers[precision] = answers
    results.append({
        "precision": precision,
        "weights_gb": round(weights_gb, 2),
        "peak_gb": round(max(peaks), 2),
        "tokens_per_sec": round(sum(speeds) / len(speeds), 1),
        "first_token_s": round(ttft, 3),
        "quality": quality_score(answers),
    })
    print("Done:", precision)

quant_df = pd.DataFrame(results)
quant_df

Now read the answers side by side. Did compression change the quality?

In [ ]:
pd.set_option("display.max_colwidth", 120)
pd.DataFrame({"prompt": TEST_PROMPTS, **{k: [a.strip() for a in v] for k, v in all_answers.items()}})

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(quant_df["precision"], quant_df["weights_gb"], color="#0f6e8c")
plt.ylabel("GPU memory for weights (GB)")
plt.title("Quantization shrinks model memory")
plt.savefig("quantization_memory.png", dpi=150, bbox_inches="tight")
plt.show()

**What to look for:** memory should drop sharply from 16-bit to 4-bit. Speed may be *slower* for 8-bit and 4-bit on a T4, because the compressed numbers have to be unpacked during calculation. That's a real finding worth writing about: **quantization mainly saves memory, not always time.**

## Phase 3: How the KV cache grows with prompt length

While generating, the model keeps a **KV cache**: its memory of every token it has already processed. Longer prompts mean a bigger cache.

We keep the 4-bit model loaded and make the prompt longer step by step.

In [ ]:
filler = "The factory recorded normal sensor readings during the morning shift. "
filler_len = len(tokenizer(filler)["input_ids"])
kv_rows = []
for target in [256, 1024, 2048, 4096, 8192]:
    long_prompt = filler * max(1, target // filler_len) + "\nSummarize the text above in one sentence."
    free_memory()
    base = torch.cuda.memory_allocated() / GB
    try:
        r = run_prompts(model, [long_prompt], max_new_tokens=32)
        n_in = len(tokenizer(long_prompt)["input_ids"])
        kv_rows.append({"prompt_tokens": n_in, "peak_gb": round(r["peak_gb"], 2),
                        "extra_gb_over_weights": round(r["peak_gb"] - base, 2),
                        "seconds": round(r["seconds"], 2)})
        print(f"{n_in} tokens -> peak {r['peak_gb']:.2f} GB")
    except torch.cuda.OutOfMemoryError:
        print(f"Out of memory at about {target} tokens")
        free_memory()
        break

kv_df = pd.DataFrame(kv_rows)
kv_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(kv_df["prompt_tokens"], kv_df["peak_gb"], marker="o", color="#0f6e8c")
plt.xlabel("Prompt length (tokens)")
plt.ylabel("Peak GPU memory (GB)")
plt.title("GPU memory grows with prompt length")
plt.grid(alpha=0.3)
plt.savefig("kv_cache_growth.png", dpi=150, bbox_inches="tight")
plt.show()

**What to look for:** the weights stay the same, but memory on top grows with every extra token. With many users and long conversations, this cache can become bigger than the model itself, which is why inference servers like vLLM manage it carefully.

## Phase 4: Batching, many prompts at once

A GPU is fastest when it has lots of parallel work. We send the same 16 prompts in batches of 1, 4, 8 and 16, and compare throughput and memory.

In [ ]:
batch_prompts = (TEST_PROMPTS * 4)[:16]
batch_rows = []
for bs in [1, 4, 8, 16]:
    free_memory()
    start = time.perf_counter(); tokens = 0; peak = 0
    for i in range(0, len(batch_prompts), bs):
        r = run_prompts(model, batch_prompts[i:i + bs])
        tokens += r["tokens"]; peak = max(peak, r["peak_gb"])
    total = time.perf_counter() - start
    batch_rows.append({"batch_size": bs, "total_seconds": round(total, 2),
                       "tokens_per_sec": round(tokens / total, 1), "peak_gb": round(peak, 2)})
    print(f"Batch {bs}: {tokens/total:.1f} tokens/sec")

batch_df = pd.DataFrame(batch_rows)
batch_df

In [ ]:
plt.figure(figsize=(7, 4))
plt.bar(batch_df["batch_size"].astype(str), batch_df["tokens_per_sec"], color="#0f6e8c")
plt.xlabel("Batch size")
plt.ylabel("Throughput (tokens per second)")
plt.title("Batching increases throughput")
plt.savefig("batching_throughput.png", dpi=150, bbox_inches="tight")
plt.show()

**What to look for:** throughput should rise a lot with bigger batches, while memory rises only a little. That's the core idea behind production inference servers: keep the GPU busy with many requests at once.

## Step 5: Save your results

Saves the tables as CSV and the charts as PNG. Download them from the **folder icon** on the left side of Colab and upload them to your GitHub repository.

In [ ]:
quant_df.to_csv("results_quantization.csv", index=False)
kv_df.to_csv("results_kv_cache.csv", index=False)
batch_df.to_csv("results_batching.csv", index=False)
print(quant_df.to_markdown(index=False)); print()
print(kv_df.to_markdown(index=False)); print()
print(batch_df.to_markdown(index=False))

## Write down what you found

Answer these in your own words. They become your README and blog post.

1. How close was the predicted weight memory to the measured memory? Why might they differ?
2. How much memory did 4-bit save compared to 16-bit? Did answer quality change?
3. Was 4-bit faster or slower than 16-bit on this GPU? Why?
4. How much extra memory did the longest prompt need?
5. How much did batching improve throughput?
6. If you had to serve this model to 100 users on one T4, which setup would you choose, and why?